In [1]:
import os
import sys
import logging
from datetime import datetime  # 1. 导入 datetime 模块
import pandas as pd

from make_ids_map_tb_utils_test import *

In [2]:
# --- 配置 ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

PROXY_URL = "http://127.0.0.1:7897"
os.environ["http_proxy"] = PROXY_URL
os.environ["https_proxy"] = PROXY_URL

class Config:
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    MISSING_DATA_MAG_DIR = 'mis_data_mag'
    DEDUPLICATE_DIR = 'deduplicate'
    
    # 输入文件路径
    RAW_DATASET = '../data/input/0_raw-dataset.csv'
 
    # 输出文件路径
    IDS_MAP_TABLE = f'{TIMESTAMP}_raw-ds_fetch-ids_add-ids.csv'
    IDS_MAP_TABLE_OUT = f'{TIMESTAMP}_out_ids-map-tb.csv'

    BYPRODUCT_TABLE = f'{TIMESTAMP}_ids-map-tb_dd_fg.csv'
    BYPRODUCT_TABLE_OUT = f'{TIMESTAMP}_out_byproduct-tb.csv'

In [3]:
# 检查输入文件
if not os.path.exists(Config.RAW_DATASET):
    logging.error(f"File not found: {Config.RAW_DATASET}")

    sys.exit(1)

logging.info("Loading dataset...")

# 2. 自动识别后缀并读取
file_ext = os.path.splitext(Config.RAW_DATASET)[1].lower()

try:
    if file_ext == '.csv':
        raw_dataset = pd.read_csv(Config.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        raw_dataset = pd.read_excel(Config.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")

# 提取唯一化合物 (去重且清理无效值)
# 这一步将 c1, c2, c3 三列展平并提取唯一值
compound_array = pd.unique(raw_dataset[['c1', 'c2', 'c3']].values.ravel())
compounds_table = pd.DataFrame(compound_array, columns=['compound']).dropna()
compounds_table = compounds_table[compounds_table['compound'].astype(str).str.strip() != '']

logging.info(f"Processing {len(compounds_table)} compounds...")

# 调用从 utils 导入的函数
# 注意：apply 返回的是 tuple 列表，后续需要拆分
results = compounds_table['compound'].apply(f_fetch_compound_info)

# 结果整合与拆分
# 将 apply 返回的 (smiles, name, cas) 元组拆分为三列
result_df = pd.DataFrame(
    results.tolist(), 
    index=compounds_table.index, 
    columns=['ismiles', 'uname', 'cas']
)
ids_map_table = pd.concat([compounds_table, result_df], axis=1)

# 保存结果
ids_map_table.to_csv(Config.IDS_MAP_TABLE, index = False)
ids_map_table.to_csv(Config.IDS_MAP_TABLE_OUT, index = False)

2026-03-10 10:21:28,341 - INFO - Loading dataset...
2026-03-10 10:21:28,355 - INFO - Processing 311 compounds...

--> 正在向 PubChem 请求: [Zinc chloride] ...
2026-03-10 10:21:34,382 - INFO -    └─ 成功获取 [smiles]: Cl[Zn]Cl
2026-03-10 10:21:34,383 - INFO -    └─ 成功获取 [uname]: Dichlorozinc
2026-03-10 10:21:34,383 - INFO -    └─ 成功获取 [cas]: 7646-85-7
2026-03-10 10:21:34,383 - INFO - Result 'Zinc chloride': SMILES=Cl[Zn]Cl, Name=Dichlorozinc, CAS=7646-85-7

--> 正在向 PubChem 请求: [1-Butyl-3-methylimidazolium chloride] ...
2026-03-10 10:21:39,584 - INFO -    └─ 成功获取 [smiles]: CCCCN1C=C[N+](=C1)C.[Cl-]
2026-03-10 10:21:39,585 - INFO -    └─ 成功获取 [uname]: 1-Butyl-3-methylimidazolium chloride
2026-03-10 10:21:39,586 - INFO -    └─ 成功获取 [cas]: 79917-90-1
2026-03-10 10:21:39,586 - INFO - Result '1-Butyl-3-methylimidazolium chloride': SMILES=CCCCN1C=C[N+](=C1)C.[Cl-], Name=1-Butyl-3-methylimidazolium chloride, CAS=79917-90-1

--> 正在向 PubChem 请求: [FeCl3] ...
2026-03-10 10:21:45,282 - INFO -    └─ 成功获取 [smi